# 🍌 BANANA FILM V2｜公開 Gemini 審查版

**新實驗分支：** `film-v2-python-owned-public-20261112`。從公開倉庫 clone，無需私人倉庫 Token；公開 main 不變。

9B 編劇 → Python 事件分段／狀態 → 9B 當段攝影 → H3 原生影音／Chain5 → 補拍與版本管理。

前端 8194；Drive 存在 `MyDrive/BananaFilmV2`。首次請用 15 秒、480p、無對白、獨立分段試拍；確認單段正常再測 Chain5。


In [ ]:
#@title ① 安裝公開 V2 實驗分支（無需私人 GitHub Token）
import os, subprocess, sys
from pathlib import Path

BRANCH='film-v2-python-owned-public-20261112'
REPO_URL='https://github.com/haohao1130-bit/banana-film-public.git'
PROJECT=Path('/content/banana-film-v2-public')
if PROJECT.exists():
    active=subprocess.check_output(['git','-C',str(PROJECT),'branch','--show-current'],text=True).strip()
    origin=subprocess.check_output(['git','-C',str(PROJECT),'remote','get-url','origin'],text=True).strip()
    if active != BRANCH or origin != REPO_URL:
        raise RuntimeError('現有資料夾不是公開 V2 分支，請建立新的 Colab 執行階段。')
    print('沿用本次執行階段的 V2 程式（更新請重啟執行階段）：', active)
else:
    subprocess.run(['git','-c','credential.helper=','clone','--depth','1',
                    '--single-branch','--branch',BRANCH,REPO_URL,str(PROJECT)],check=True)
assert (PROJECT/'film_v2'/'service.py').is_file(), 'V2 引擎未安裝'
sys.path.insert(0,str(PROJECT/'tools'))
import film_v2_runtime as runtime
print('🔎 執行 Python 語法與事件核心回歸測試')
subprocess.run([sys.executable,'-m','compileall','-q','film_v2'],cwd=PROJECT,check=True)
test_env=dict(os.environ,BANANA_DATA=str(PROJECT/'.v2_test_data'))
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_film_v2_production.py','-v'],
               cwd=PROJECT,env=test_env,check=True)
runtime.install(globals())
print('🔎 執行整合模擬測試')
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_film_v2_integration.py','-v'],
               cwd=PROJECT,env=test_env,check=True)
print('✅ Python 回歸測試全部執行完畢；接著可下載模型')


In [ ]:
#@title ② 下載 H3 和 9B 模型
USE_DRIVE_CACHE = False #@param {type:"boolean"}
runtime.download(globals(),use_drive_cache=USE_DRIVE_CACHE)


In [ ]:
#@title ③ 啟動 V2 製片所（8194）
SAVE_TO_DRIVE = True #@param {type:"boolean"}
runtime.launch(globals(),save_to_drive=SAVE_TO_DRIVE)


## 🎬 操作與 Gemini 審查重點

① 編劇預覽：9B 交原子事件、穩定人物 ID、台詞及明確狀態變化。② Python 分槽：合法 H3 影格、事件不重演、跨段狀態延續。③ H3 原生影音：首段獨立試跑，第二次測兩段 Chain5、音畫裁切。④ 劇本審查、失敗續跑、補拍選版與保存。

審查入口：`docs/FILM_V2_GEMINI_REVIEW_20261112.md`。目前未宣稱 Colab GPU 端到端驗證通過。作品儲存位置 `MyDrive/BananaFilmV2/films_v2`，與舊版隔離。


In [ ]:
#@title 🔍 診斷紀錄
subprocess.run(['nvidia-smi'])
for name in ('comfyui','banana_film_v2'):
    log=Path('/content')/(name+'.log')
    print(name, log.read_text(errors='replace')[-6000:] if log.exists() else '尚無紀錄')


In [ ]:
#@title 🧯 手動停止 ComfyUI
runtime.show_shutdown_control(globals())
